# Training on soft labels: typed-decisions

[LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) has
1,200 training and 400 test records from four operational workflows: agent-trace monitoring,
customer service, invoice processing and security incidents. Each record is a structured JSON
state with the same five questions for its workflow.

Its labels are **soft**: each answer is a probability spread from several raters, for example
`{"continue": 0.28, "human_review": 0.43, "observe": 0.25, "stop": 0.03}`. sysone trains on these
distributions directly (passed as `gold`) and evaluates against them, so accuracy here means
"picks the raters' top answer", and **distance** shows how close the predicted spread is.

It trains ModernBERT-base, a bidirectional encoder, with `tuning="head"`, then with `tuning="lora"` on a subset. It runs faster than [`decoder_typed_decisions.ipynb`](decoder_typed_decisions.ipynb), the same notebook
with Qwen3-0.6B, and the two can be compared directly.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from GitHub.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone @ git+https://github.com/ForwardMetric/sysone@milestone-2-backbone-tuning"

In [ ]:
import json

from datasets import load_dataset

from sysone import EncoderModel, baselines, compare

N_TRAIN_LORA = 100  # records for the LoRA section, which trains the backbone and is much slower
LORA_EPOCHS = 2

## Load the data

`state`, `questions` and `gold` are JSON text, which sysone decodes itself. The workflow is used as
the row's `category`.

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))
print("rater probabilities for one question:", json.loads(train[0]["gold"])["action"]["probabilities"])

## Before training: baselines

- **random guess**: every option equally likely.
- **average answer**: for each workflow question, the raters' average distribution in the
  training data. Every record in a workflow has the same questions, so this is a strong bar: a
  model that beats it is reading each record, not just learning the typical answer.

In [ ]:
results = baselines(test, train=train)
compare(results)

## Train

The soft `gold` distributions are the training targets. 10% of records are held back for
validation and early stopping, split evenly across workflows.

In [ ]:
model = EncoderModel(tuning="head")
model.fit(train)

## After training

In [ ]:
results["head"] = model.evaluate(test)
compare(results)

## Fine-tuning the backbone with LoRA

With `tuning="lora"`, small adapters on the top 60% of the kept layers train along with the head,
so the backbone itself adapts to these records. Nothing can be cached, since the backbone's
outputs change every step: each example costs a forward and a backward pass through the backbone
every epoch. That is far slower than head-only training, so this section trains on the first
`N_TRAIN_LORA` records only. A head-only model on the same records gives the fair comparison.

Validation runs about ten times per epoch, and training stops early if it stops improving.
ModernBERT-base is about a quarter of Qwen3-0.6B's size, so this runs faster than in the
decoder version of this notebook. A CUDA GPU is recommended for
full-size runs, and there `tuning="qlora"` (the default) uses less memory.

In [ ]:
small = train[:N_TRAIN_LORA]
results["head, %d records" % N_TRAIN_LORA] = EncoderModel(tuning="head", verbose=0).fit(small).evaluate(test)

lora = EncoderModel(tuning="lora", epochs=LORA_EPOCHS)
lora.fit(small)
results["lora, %d records" % N_TRAIN_LORA] = lora.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).

With soft labels, **distance** and **log_loss** matter as much as accuracy: when raters split
45/55, either answer is defensible, but the probabilities should reflect the split.

In [ ]:
compare(results, by="overall")

### By workflow

In [ ]:
compare(results, by="category")

## Reading the results

- Raters often disagree, so accuracy has a ceiling well below 100%.
- **average answer** is the bar to beat. With a frozen backbone, the head has limited room to get
  past it. Fine-tuning the backbone with LoRA is what should move these numbers further; compare
  it with head-only on the same records, and expect the gap to grow with more records and epochs.
- Probabilities are not calibrated yet (temperatures stay at 1.0), which affects **distance** and
  **log_loss** more than accuracy.
- A long LoRA run can be stopped with Ctrl-C and continued later with `lora.fit(small)`, from
  where it stopped.